## Init config

In [3]:
import torch
from common_functions_python import set_config_file, test_function
config_file = {
                'name': 'DINO_left_hand',
                'datasets': ['dino_left_large'],
                'bidirectional_lstm': False,
                'lstm_dropout': 0.3,
                'mlp_dropout': 0.3,
                'lr': 0.0002,
                'step_size': 5,
                'gamma': 0.5,
                'weight_decay': 0,
                'hidden_dim': 512,
                'num_layers': 3,
                'batch_size': 64, 
                'frame_frequency': 4,
                'num_workers': 4,
                'num_epoch': 30
                }

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print('device: ', device)
set_config_file(config_file, device)
# test_function()

device:  cuda


## Create dataset

In [4]:
from common_functions_python import create_data_loaders, create_model, create_train_dependencies

train_loader, test_loader = create_data_loaders(config_file['datasets'])

input_dim = train_loader.dataset[0][0][0].size(0)  # Get input dimension from a single feature from a video
num_classes = len(set(train_loader.dataset.classes))
print("input_dim: ", input_dim, " num_classes: ", num_classes)
print("train_dataset size: ", len(train_loader.dataset))
print("test_dataset size: ", len(test_loader.dataset))

model = create_model(input_dim, num_classes)

criterion, optimizer, scheduler = create_train_dependencies(model)

input_dim:  1024  num_classes:  744
train_dataset size:  18018
test_dataset size:  4524


## Train

In [5]:
from tqdm import tqdm
from common_functions_python import get_current_time, plot_result, save_model_result, test_model
from torch.nn.utils.rnn import pack_padded_sequence


print(f"lr {config_file['lr']}, step_size: {config_file['step_size']}, gamma: {config_file['gamma']}, weight_decay: {config_file['weight_decay']}")
print(f"Model hidden_dim {config_file['hidden_dim']}, num_layers: {config_file['num_layers']}")
print(f"batch_size {config_file['batch_size']}, frame_frequency: {config_file['frame_frequency']}")

avg_loss_list = []
avg_accuracy_list = []
avg_test_accuracy_list = []
avg_top5_test_accuracy_list = []
avg_test_loss_list = []

num_epoch = config_file['num_epoch']
for epoch in range(1, num_epoch+1):
    loop = tqdm(train_loader)
    running_loss = 0.0
    running_accuracy= 0.0
    for idx, (features, lengths, labels) in enumerate(loop):
        packed_input = pack_padded_sequence(features, lengths, batch_first=True, enforce_sorted=True)

        # features = features.unsqueeze(-1).float().to(device)
        features = packed_input.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)

        predictions = outputs.argmax(dim=1, keepdim=True).squeeze()
        correct = (predictions == labels).sum().item()
        accuracy = correct / len(labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_accuracy += 100 * accuracy
        loop.set_description(f"Epoch [{epoch}/{num_epoch}]")
        loop.set_postfix(loss=loss.item(), acc=accuracy)
    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    avg_accuracy = running_accuracy / len(train_loader)
    print(f"Time: {get_current_time()} Epoch [{epoch}], Avg loss: {avg_loss:.4f}, Avg accuracy: {avg_accuracy:.4f}")
    avg_test_accuracy, avg_top5_test_accuracy, avg_test_loss = test_model(test_loader, model, criterion)

    avg_loss_list.append(avg_loss)
    avg_accuracy_list.append(avg_accuracy)
    avg_test_accuracy_list.append(avg_test_accuracy)
    avg_top5_test_accuracy_list.append(avg_top5_test_accuracy)
    avg_test_loss_list.append(avg_test_loss)

plot_result(avg_accuracy_list, avg_test_accuracy_list, avg_top5_test_accuracy_list, avg_loss_list, avg_test_loss_list)
current_time = get_current_time()
save_model_result(model, current_time, input_dim, num_classes, avg_accuracy_list, avg_test_accuracy_list, avg_top5_test_accuracy_list, avg_loss_list, avg_test_loss_list)

lr 0.0002, step_size: 5, gamma: 0.5, weight_decay: 0
Model hidden_dim 512, num_layers: 3
batch_size 64, frame_frequency: 4


Epoch [1/30]:  74%|███████▍  | 208/282 [00:05<00:01, 40.54it/s, acc=0.0625, loss=5.45]


KeyboardInterrupt: 